### Utility functions for taking input of samples

In [1]:
def input_data(n=None, dtype=float):
    data = []
    i = 0
    while True:
        if i==n: return data
        inp = input(f'Enter element {i+1}: ')
        if not inp and n is None: return data
        data.append(dtype(inp))
        i += 1

def input_table(r=None, c=None, dtype=float, ragged=False):
    print('Row 1:')
    row1 = input_data(c, dtype=dtype)
    table = [row1]
    c_new = c if ragged else len(row1)
    while True:
        try:
            if len(table)==r: return table
            print(f'Row {len(table)+1}:')
            row = input_data(c_new, dtype=dtype)
            if not row and r is None: return table
            table.append(row)
        except ValueError as e:
            if r is c is None: return table
            else: raise e

## 6a: Write a program to implement testing of hypothesis for contingency tables using ꭓ² test.

In [2]:
from scipy.stats import chi2
import numpy as np
from math import log

def chi2test(table, lambda_=1):
    func = (lambda O, E: 2*O*log(O/E), lambda O, E: (O-E)**2/E)[lambda_]
    table = np.asarray(table)
    margin_row = table.sum(axis=0)
    margin_col = table.sum(axis=1)
    N = margin_col.sum()
    CHI2 = 0
    for i, row in enumerate(table):
        for j, O in enumerate(row):
            E = margin_col[i]*margin_row[j]/N
            CHI2 += func(O, E)
    
    return CHI2, chi2.sf(CHI2, (table.shape[0]-1)*(table.shape[1]-1))

In [3]:
table = input_table()
correct = bool(input('Should the continuity correction be applied?: '))

results_pearsons = chi2test(table, lambda_=1)
results_gtest = chi2test(table, lambda_=0)

Row 1:
Row 2:
Row 3:
Row 4:
Row 5:
Row 6:


In [4]:
print('Sample:', table)
print('continuity correction:', correct)

Sample: [[94.0, 93.0, 39.0, 29.0], [78.0, 100.0, 34.0, 46.0], [23.0, 30.0, 19.0, 58.0], [33.0, 24.0, 10.0, 72.0], [54.0, 21.0, 85.0, 57.0]]
continuity correction: False


In [5]:
results_pearsons

(210.9662189083329, 1.765550729359331e-38)

In [6]:
results_gtest

(205.27774005513342, 2.65063134436974e-37)

In [7]:
from scipy.stats import chi2_contingency
results_pearsons = chi2_contingency(table, correction=correct, lambda_=1)
results_gtest = chi2_contingency(table, correction=correct, lambda_=0)

In [8]:
print('statistic={0.statistic}, pvalue={0.pvalue}, df={0.dof}'.format(results_pearsons))
print(results_pearsons.expected_freq)

statistic=210.9662189083329, pvalue=1.765550729359331e-38, df=12
[[71.98198198 68.40840841 47.73273273 66.87687688]
 [72.82882883 69.21321321 48.29429429 67.66366366]
 [36.6966967  34.87487487 24.33433433 34.09409409]
 [39.23723724 37.28928929 26.01901902 36.45445445]
 [61.25525526 58.21421421 40.61961962 56.91091091]]


In [9]:
print('statistic={0.statistic}, pvalue={0.pvalue}, df={0.dof}'.format(results_gtest))
print(results_gtest.expected_freq)

statistic=205.27774005513336, pvalue=2.6506313443698147e-37, df=12
[[71.98198198 68.40840841 47.73273273 66.87687688]
 [72.82882883 69.21321321 48.29429429 67.66366366]
 [36.6966967  34.87487487 24.33433433 34.09409409]
 [39.23723724 37.28928929 26.01901902 36.45445445]
 [61.25525526 58.21421421 40.61961962 56.91091091]]


## 6b: Write a program to implement testing of hypothesis for 2x2 contingency tables using McNemar's test.

In [2]:
from scipy.stats import chi2

def mcnemarstest(table2x2, correction=False):
    (_, b), (c, _) = table2x2
    cc = 1 if correction else 0
    CHI2 = (abs(b-c)-cc)**2/(b+c)
    return CHI2, chi2.sf(CHI2, 1)

In [3]:
table2x2 = input_table(2, 2)
correct = bool(input('Should the continuity correction be applied?: '))

results = mcnemarstest(table2x2, correction=correct)

Row 1:
Row 2:


In [5]:
print('Sample:', table2x2)
print('continuity correction:', correct)
results

Sample: [[3.0, 6.0], [1.0, 9.0]]
continuity correction: False


(3.5714285714285716, 0.05878172135535891)

## 6c: Write a program to implement testing of hypothesis for 2x2 contingency tables using Fisher's exact test.

In [6]:
from math import factorial

def _f(a, b, c, d):
    return (factorial(a+b)*factorial(c+d)*factorial(a+c)*factorial(b+d)) / (factorial(a+b+c+d)*factorial(a)*factorial(b)*factorial(c)*factorial(d))

def fishersexacttest(table2x2, alternative='two-sided'):
    if alternative not in {'two-sided', 'less', 'greater'}:
        raise ValueError("alternative must be 'less', 'greater' or 'two-sided'")
    
    (a, b), (c, d) = table2x2
    Ps = [pa:=_f(a, b, c, d)]
    if alternative!='greater':
        while a>0<d:
            a -= 1
            b += 1
            c += 1
            d -= 1
            Ps.append(_f(a, b, c, d))
    
    (a, b), (c, d) = table2x2
    if alternative!='less':
        while b>0<c:
            a += 1
            b -= 1
            c -= 1
            d += 1
            Ps.append(_f(a, b, c, d))
    
    return sum(p for p in Ps if p<=pa) if alternative=='two-sided' else sum(Ps)

In [7]:
table2x2 = input_table(2, 2, dtype=int)
alt = alt if (alt:=input('Enter the type of test: ')) else 'two-sided'

results = fishersexacttest(table2x2, alternative=alt)

Row 1:
Row 2:


In [8]:
print('Sample:', table2x2)
print('alternative:', alt)
results

Sample: [[2, 4], [3, 7]]
alternative: two-sided


1.0

### Using scipy.stats module

In [9]:
from scipy.stats import fisher_exact
table2x2 = input_table(2, 2, dtype=int)
alt = alt if (alt:=input('Enter the type of test: ')) else 'two-sided'

results = fisher_exact(table2x2, alternative=alt)

Row 1:
Row 2:


In [10]:
print('Sample:', table2x2)
print('alternative:', alt)
results

Sample: [[2, 4], [3, 7]]
alternative: two-sided


SignificanceResult(statistic=1.1666666666666667, pvalue=1.0)

## 6d: Write a program to implement testing of hypothesis for goodness of fit using ꭓ² test.

In [11]:
import scipy.stats
from math import log

def chi2test(obs, exp=None, pf=None, args=(), proportions=False, lambda_=1):
    func = (lambda O, E: 2*O*log(O/E), lambda O, E: (O-E)**2/E)[lambda_]
    N = sum(obs)
    if pf is not None:
        if isinstance(pf, str):
            dist, pf = pf.split('.')
            pf = getattr(getattr(scipy.stats, dist), pf)
        exp = pf(exp, *args)*N
    elif exp is None:
        exp = [N/len(obs)]*len(obs)
    elif proportions:
        p = N/sum(exp)
        exp = [p*prop for prop in exp]
    
    return (CHI2:=sum(func(O, E) for O, E in zip(obs, exp, strict=True))), scipy.stats.chi2.sf(CHI2, len(obs)-1)

In [21]:
print('Observed frequencies:')
print(obs:=input_data())
print('Distribution function:')
print(pf:=pf if (pf:=input('Enter the probability function name: ')) else None)
if pf is not None:
    props = False
    print('Distribution parameters:')
    print(params:=input_data())
    print('Random variables:')
else:
    params = ()
    print(props:=bool(input('Will the entered expected frequencies be proportions?: ')))
    print('Expected frequencies:')
print(exp:=input_data(len(obs)))

results_pearsons = chi2test(obs, exp, pf, params, props, lambda_=1)
results_gtest = chi2test(obs, exp, pf, params, props, lambda_=0)

Observed frequencies:
[1.0, 2.0, 7.0, 9.0]
Distribution function:
binom.pmf
Distribution parameters:
[5.0, 0.5]
Random variables:
[0.0, 1.0, 2.0, 3.0]


In [22]:
results_pearsons

(2.3638157894736853, 0.5004058411983157)

In [23]:
results_gtest

(9.254181673585126, 0.026095505968010652)